# InSAR Tĩnh Túc — Interactive Analysis Notebook

Notebook này cho phép chạy và khám phá từng bước của pipeline một cách tương tác.

## Cấu trúc:
1. **Setup** — Import và cấu hình
2. **Phase 2** — P-SBAS velocity maps + MAC clustering
3. **Phase 3** — 4D movements tại điểm P2
4. **Phase 4** — Kinematics (strain, thickness, WTC)
5. **Custom** — Phân tích tùy chỉnh


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Setup complete')

## 1. Chạy Phase 2: P-SBAS + Clustering

In [ ]:
from run_pipeline import run_phase_1_data_preparation, run_phase_2_sbas_clustering
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

hydro_data, hydro_dates = run_phase_1_data_preparation()
vel_asc, vel_desc, ts, dates, dem, slope, macs = run_phase_2_sbas_clustering(H=100, W=100)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (vel, title) in zip(axes, [(vel_asc, 'Ascending'), (vel_desc, 'Descending')]):
    im = ax.imshow(vel, cmap='RdYlGn_r', vmin=-30, vmax=10, aspect='auto')
    plt.colorbar(im, ax=ax, label='mm/yr')
    ax.set_title(f'LOS Velocity — {title}\nTĩnh Túc, Cao Bằng')
plt.tight_layout()
plt.show()
print(f'MACs found: {len(macs)}')
for m in macs[:3]:
    print(f"  MAC {m['mac_id']}: {m['classification']}, vel={m.get('mean_velocity_mm_yr',0):.1f}mm/yr, risk={m.get('risk_score',0)}")

## 2. Điều chỉnh ngưỡng phân cụm

In [ ]:
from src.clustering.spatial_clustering import SpatialClusterer
from src.classification.mac_classifier import MACClassifier, generate_synthetic_ancillary
from src.utils.geo_utils import decompose_2d, compute_slope

# Thử các ngưỡng khác nhau
for threshold in [0.5, 1.0, 1.5, 2.0]:
    c = SpatialClusterer(threshold, 2, 3, 80.0)
    m = c.cluster(vel_asc)
    print(f'  Threshold {threshold} cm/yr: {len(m)} MACs')

## 3. 4D Monitoring tại hotspot P2

In [ ]:
from run_pipeline import run_phase_3_fusion_4d
results_4d = run_phase_3_fusion_4d(dem, slope, dates, hydro_data, hydro_dates)

In [ ]:
# Vẽ chuỗi thời gian tại P2
p2 = results_4d.get('P2', {})
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
comps = [('east', 'East (mm)', '#2196F3'), ('north', 'North (mm)', '#4CAF50'),
         ('vertical', 'Vertical (mm)', '#F44336')]
for ax, (comp, label, color) in zip(axes, comps):
    data = p2.get(comp, [])
    if len(data):
        ax.plot(data, color=color, linewidth=1.5, label=f'P2 — {comp}')
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_ylabel(label)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Days since monitoring start')
fig.suptitle('Daily 4D Movements — Hotspot P2 (Talus Slope)\nTĩnh Túc, Cao Bằng',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Kinematics: Strain invariants + Thickness

In [ ]:
from src.kinematics.kinematics_analyzer import StrainAnalyzer, SlipSurfaceInverter

H, W = dem.shape
x, y = np.meshgrid(np.linspace(0,1,W), np.linspace(0,1,H))
rng = np.random.default_rng(42)
decay = np.exp(-((x-0.35)**2 + (y-0.35)**2)/0.04)
ve_field = -25.0 * decay + rng.normal(0, 0.5, (H,W))
vn_field =  -8.0 * decay + rng.normal(0, 0.3, (H,W))
vv_field = -15.0 * decay + rng.normal(0, 0.3, (H,W))

sa = StrainAnalyzer()
strain = sa.compute_strain_tensor(ve_field, vn_field, vv_field)

inv = SlipSurfaceInverter()
thickness = inv.estimate_thickness(ve_field, vn_field, vv_field, dem)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (data, title, cmap) in zip(axes, [
    (strain['mss'], 'Max Shear Strain (×10⁻³)', 'hot_r'),
    (strain['dil'], 'Dilatation (×10⁻³)', 'RdBu_r'),
    (thickness, 'Estimated Thickness (m)', 'YlOrRd')
]):
    vabs = np.nanpercentile(np.abs(data), 98)
    vm = (-vabs, vabs) if 'Dil' in title else (0, vabs)
    im = ax.imshow(data, cmap=cmap, vmin=vm[0], vmax=vm[1], aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=10)
plt.suptitle('Kinematics Analysis — Tĩnh Túc', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Max thickness: {np.nanmax(thickness):.1f}m')

## 5. Phân tích tùy chỉnh — Wavelet Coherence

In [ ]:
from src.kinematics.kinematics_analyzer import TemporalAnalyzer

n_days = 365 * 3
t = np.arange(n_days)
# Mô phỏng: displacement phản hồi theo lượng mưa với độ trễ ~15 ngày
rain_signal = np.sin(2*np.pi*t/365) * 15 + np.random.default_rng(0).normal(0, 3, n_days)
delay = 15
displ_signal = np.convolve(rain_signal, np.ones(delay)/delay, mode='same') * -0.3

analyzer = TemporalAnalyzer()
wtc, periods = analyzer.wavelet_coherence(displ_signal, rain_signal, dt=1.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
ax1.plot(t, rain_signal, 'b-', alpha=0.7, label='Rainfall proxy')
ax1_r = ax1.twinx()
ax1_r.plot(t, displ_signal, 'r-', linewidth=1.5, label='Displacement (mm)')
ax1.set_ylabel('Rainfall (mm/day)', color='b')
ax1_r.set_ylabel('Displacement (mm)', color='r')
ax1.set_title('Input time series')
ax1.grid(True, alpha=0.3)

im = ax2.pcolormesh(t, periods, wtc, cmap='hot_r', vmin=0, vmax=1, shading='auto')
ax2.set_yscale('log')
ax2.axhline(365, color='cyan', linewidth=2, linestyle='--', label='1-year period')
plt.colorbar(im, ax=ax2, label='WTC coherence')
ax2.set_ylabel('Period (days)')
ax2.set_xlabel('Time (days)')
ax2.set_title('Wavelet Transform Coherence: Displacement vs Rainfall')
ax2.legend()
plt.tight_layout()
plt.show()

annual_idx = np.argmin(np.abs(periods - 365))
print(f'Annual WTC (displacement ~ rainfall): {np.nanmean(wtc[annual_idx]):.2f}')